In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [111]:
import pandas as pd
import numpy as np
import sqlalchemy as sa
import os
from row_generate_func import *
from auto_insert_func import *
from sqlalchemy.orm import Session

engine = sa.create_engine("postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)

In [24]:
# --- Main workflow ---
output_folder = '/workspaces/crmprtd/fern/v4_after_metadata_update/output/'
file_name = 'Fern_raw_db_station_matched.csv'

df_match = pd.read_csv(output_folder + file_name)

df_match

,station_name,native_id_raw,station_file_name,lat,lon,elev,variable,unit_raw,earliest_time_raw,latest_time_raw,db_var_match,unit_db,earliest_time_db,latest_time_db,earliest_diff_days,latest_diff_days,batch
0,Atlin School,AtlSch,Atlin school,59.574000,-133.687000,705,Air Temp,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,TempC,celsius,NaN,NaN,NaN,NaN,batch4
1,Atlin School,AtlSch,Atlin school,59.574000,-133.687000,705,Air Temp 2,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,NaN,NaN,NaN,NaN,NaN,NaN,batch5
2,Atlin School,AtlSch,Atlin school,59.574000,-133.687000,705,DewPt,°C,2011-07-28 09:00:00,2024-08-10 19:00:00,DewPtC,celsius,NaN,NaN,NaN,NaN,batch2
3,Atlin School,AtlSch,Atlin school,59.574000,-133.687000,705,Gd Temp 25 cm,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,Soil_Temp_25cm,celsius,NaN,NaN,NaN,NaN,batch4
4,Atlin School,AtlSch,Atlin school,59.574000,-133.687000,705,Gd Temp 50 cm,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,Soil_Temp_50cm,celsius,NaN,NaN,NaN,NaN,batch4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,Willow-BowronWx,PGTIS2,WillowBowron PGTIS 2,53.772577,-122.724634,596,Solar Radiation,W/m²,2007-07-31 15:00:00,2024-10-24 09:00:00,SolarRadiationWm,W m-2,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,373.0,batch1
423,Willow-BowronWx,PGTIS2,WillowBowron PGTIS 2,53.772577,-122.724634,596,Temp,°C,2007-07-31 15:00:00,2024-10-24 09:00:00,TempC,celsius,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,373.0,batch1
424,Willow-BowronWx,PGTIS2,WillowBowron PGTIS 2,53.772577,-122.724634,596,Wetness,%,2008-06-12 13:00:00,2024-10-24 09:00:00,Wetness,%,2008-06-12 13:00:00,2023-10-17 09:00:00,0.0,373.0,batch1
425,Willow-BowronWx,PGTIS2,WillowBowron PGTIS 2,53.772577,-122.724634,596,Wind Direction,ø,2007-07-31 15:00:00,2024-10-24 09:00:00,WindDirection,degree,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,373.0,batch1


In [25]:
station_names = df_match['station_name'].tolist()

query = sa.text("""
    SELECT DISTINCT m.station_name, s.native_id, m.lat, m.lon, m.elev
    FROM meta_history AS m
    JOIN meta_station AS s ON m.station_id = s.station_id
    WHERE s.network_id = 11
      AND m.station_name = ANY(:station_names)
""")

with engine.connect() as conn:
    db_stations = pd.read_sql(query, conn, params={"station_names": station_names})

In [26]:
merged = df_match.merge(
    db_stations,
    on='station_name',
    how='left'
)

match_info = merged.rename(
    columns={
        'native_id': 'native_id_db',
        'lat_y': 'lat_db',
        'lon_y': 'lon_db',
        'elev_y': 'elev_db'
    }
)

match_info = match_info.drop(columns=['native_id_raw', 'lat_x', 'lon_x', 'elev_x'])



In [27]:
match_info[['station_name', 'variable']]

,station_name,variable
0,Atlin School,Air Temp
1,Atlin School,Air Temp 2
2,Atlin School,DewPt
3,Atlin School,Gd Temp 25 cm
4,Atlin School,Gd Temp 50 cm
...,...,...
422,Willow-BowronWx,Solar Radiation
423,Willow-BowronWx,Temp
424,Willow-BowronWx,Wetness
425,Willow-BowronWx,Wind Direction


In [72]:
import re

# 1️⃣ Define base variable
def base_variable(var):
    return re.sub(r'\s*\d+$', '', var).strip()

match_info['base_variable'] = match_info['variable'].apply(base_variable)

# 2️⃣ Filter rows with duplicates per station + base_variable + unit_raw
dup_items = (
    match_info
    .groupby(['station_name', 'base_variable', 'unit_raw'])
    .filter(lambda x: len(x) > 1)
    .reset_index(drop=True)
)

dup_items

,station_name,station_file_name,variable,unit_raw,earliest_time_raw,latest_time_raw,db_var_match,unit_db,earliest_time_db,latest_time_db,earliest_diff_days,latest_diff_days,batch,native_id_db,lat_db,lon_db,elev_db,base_variable
0,Atlin School,Atlin school,Air Temp,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,TempC,celsius,NaN,NaN,NaN,NaN,batch4,AtlSch,59.574000,-133.687000,705.0,Air Temp
1,Atlin School,Atlin school,Air Temp 2,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,NaN,NaN,NaN,NaN,NaN,NaN,batch5,AtlSch,59.574000,-133.687000,705.0,Air Temp
2,BowronPit,Bowron Pit,DewPt,°C,2018-09-13 12:00:00,2025-01-07 11:00:00,DewPtC,celsius,2018-09-13 12:00:00,2024-01-05 08:00:00,0.0,368.0,batch1,BowronPit,53.902111,-122.015389,722.0,DewPt
3,BowronPit,Bowron Pit,DewPt 2,°C,2018-08-29 15:00:00,2024-10-16 13:00:00,NaN,NaN,NaN,NaN,NaN,NaN,batch5,BowronPit,53.902111,-122.015389,722.0,DewPt
4,BowronPit,Bowron Pit,RH,%,2018-09-13 12:00:00,2025-01-07 11:00:00,RH,%,2018-08-29 15:00:00,2024-01-05 08:00:00,14.0,368.0,batch1,BowronPit,53.902111,-122.015389,722.0,RH
5,BowronPit,Bowron Pit,RH 2,%,2018-08-29 15:00:00,2024-10-16 13:00:00,NaN,NaN,NaN,NaN,NaN,NaN,batch5,BowronPit,53.902111,-122.015389,722.0,RH
6,BowronPit,Bowron Pit,Temp,°C,2018-09-13 12:00:00,2025-01-07 11:00:00,TempC,celsius,2018-08-29 15:00:00,2024-01-05 08:00:00,14.0,368.0,batch1,BowronPit,53.902111,-122.015389,722.0,Temp
7,BowronPit,Bowron Pit,Temp 2,°C,2018-08-29 15:00:00,2024-10-16 13:00:00,NaN,NaN,NaN,NaN,NaN,NaN,batch5,BowronPit,53.902111,-122.015389,722.0,Temp
8,Caribou Pass,Caribou Pass,DewPt,?C,2023-08-04 11:00:00,2025-01-07 14:00:00,DewPtC,celsius,2023-08-04 11:00:00,2023-12-14 06:00:00,0.0,390.0,batch1,CaribouPass,58.357180,-129.546370,1744.0,DewPt
9,Caribou Pass,Caribou Pass,DewPt 2,?C,2023-08-04 11:00:00,2025-01-07 14:00:00,NaN,NaN,NaN,NaN,NaN,NaN,batch5,CaribouPass,58.357180,-129.546370,1744.0,DewPt


2 sensors for the same station variable, the first sensors value has been inserted. Now for more records have only the second sensor or both, I insert them or update them. For the item with only the second sensor, seemes insert successufully. So need to update the values only from one sensor to the mean of two sensors. So I need to update the values of that obs_raw values. Right now I just need to update the obs_raw.

In [76]:
df_data['sensor_source'].unique()

array(['DewPt 2, ?C', 'mean of DewPt, ?C, DewPt 2, ?C'], dtype=object)

In [92]:
dup_rows = []
data_path = '/fern_data/FERNNorth2024_VF/WxData24'
network_name = "FLNRO-FERN"

stations = dup_items.groupby(['station_name', 'base_variable'])

for (station_name, base_var), group in stations:
    
    print(f"Processing station: {station_name}, variable: {base_var}")

    csv_file_name = group['station_file_name'].iloc[0] + '.csv'
    csv_path = os.path.join(data_path, csv_file_name)
    df_data = pd.read_csv(csv_path, encoding='latin1', low_memory=False)

    time_col = 'Date' if 'Date' in df_data.columns else df_data.columns[0]
    df_data[time_col] = pd.to_datetime(df_data[time_col], errors='coerce')

    unit_raw = group['unit_raw'].iloc[0]
    variable_cols = [f"{v}, {unit_raw}" for v in group['variable'] if f"{v}, {unit_raw}" in df_data.columns]

    if not variable_cols or len(variable_cols) < 2:
        # Skip if only one sensor exists
        print(f"  ⚠ Only one sensor for {base_var}, skipping.")
        continue

    # Compute row-wise mean across sensors
    df_data['value'] = df_data[variable_cols].mean(axis=1, skipna=True)

    # Create a column showing which sensors contributed
    def contributing_sensors(row):
        sensors = [col for col in variable_cols if pd.notna(row[col])]
        if len(sensors) > 1:
            return f"mean of {', '.join(sensors)}"
        elif sensors:
            return sensors[0]
        else:
            return None

    df_data['sensor_source'] = df_data.apply(contributing_sensors, axis=1)

    # --- Filter: skip rows that only come from the first sensor ---
    # Assume the first sensor column is variable_cols[0]
    df_data = df_data[~(df_data['sensor_source'] == variable_cols[0])]
    df_data = df_data.dropna(subset=['value']).reset_index(drop=True)

    # --- Check for differences between sensors ---
    # Only consider rows where both sensor values exist
    sensor_diff_rows = df_data.dropna(subset=variable_cols)
    
    # Count how many rows have different values across sensors
    # Use `nunique` row-wise
    sensor_diff_count = (sensor_diff_rows[variable_cols].nunique(axis=1) > 1).sum()
    
    print(f"  🔹 Rows with differing sensor values: {sensor_diff_count} out of {len(sensor_diff_rows)}")

    print(f"{len(df_data)} values from {df_data['sensor_source'].unique()}")

    if df_data.empty:
        print(f"  ⚠ No rows left after skipping first sensor for {base_var}")
        continue

    # --- Build Row objects ---
    for _, row_i in df_data.iterrows():
        dup_rows.append(Row(
            time=row_i[time_col],
            val=row_i['value'],
            variable_name=group['db_var_match'].iloc[0],
            unit=group['unit_db'].iloc[0],
            network_name=network_name,
            station_id=group['native_id_db'].iloc[0],
            lat=float(group['lat_db'].iloc[0]),
            lon=float(group['lon_db'].iloc[0])
        ))

Processing station: Atlin School, variable: Air Temp
  🔹 Rows with differing sensor values: 108905 out of 119192
119192 values from ['mean of Air Temp, °C, Air Temp 2, °C']
Processing station: BowronPit, variable: DewPt
  🔹 Rows with differing sensor values: 50033 out of 50049
50411 values from ['DewPt 2, °C' 'mean of DewPt, °C, DewPt 2, °C']
Processing station: BowronPit, variable: RH
  🔹 Rows with differing sensor values: 48461 out of 50049
50411 values from ['RH 2, %' 'mean of RH, %, RH 2, %']
Processing station: BowronPit, variable: Temp
  🔹 Rows with differing sensor values: 50407 out of 50719
51081 values from ['Temp 2, °C' 'mean of Temp, °C, Temp 2, °C']
Processing station: Caribou Pass, variable: DewPt
  🔹 Rows with differing sensor values: 6812 out of 6815
12531 values from ['mean of DewPt, ?C, DewPt 2, ?C' 'DewPt 2, ?C']
Processing station: Caribou Pass, variable: RH
  🔹 Rows with differing sensor values: 6703 out of 6815
12531 values from ['mean of RH, %, RH 2, %' 'RH 2, %']

In [88]:
db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"

safe_insert_rows(
    dup_rows,
    log_file_path=output_folder + 'fern_all_station_insert.log',
    db_url=db_url,
    chunk_size=5000,        # SAFE starting point
    bulk_chunk_size=1000,   # internal DB bulk insert size
)

🚀 Starting insert: 804066 rows in 161 chunks
➡️  Processing rows 0–4999
{"asctime": "2025-12-17 22:56:33,849", "levelname": "INFO", "name": "crmprtd.insert", "message": "Using Bulk Insert Strategy"}
{"asctime": "2025-12-17 22:56:34,004", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 0 inserted, 1000 skipped, 0 failed"}
{"asctime": "2025-12-17 22:56:34,073", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 0 inserted, 2000 skipped, 0 failed"}
{"asctime": "2025-12-17 22:56:34,138", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 0 inserted, 3000 skipped, 0 failed"}
{"asctime": "2025-12-17 22:56:34,221", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 0 inserted, 4000 skipped, 0 failed"}
{"asctime": "2025-12-17 22:56:34,288", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 0 inserted, 5000 skipped, 0 failed"}
{"asctime": "202

In [107]:
update_sql = sa.text("""
UPDATE obs_raw AS o
SET datum = :val
FROM meta_history AS m,
     meta_station AS s,
     meta_vars AS v
WHERE o.history_id = m.history_id
  AND m.station_id = s.station_id
  AND o.vars_id = v.vars_id
  AND s.native_id = :station_id
  AND v.net_var_name = :variable_name
  AND o.obs_time = :obs_time
  AND o.datum IS DISTINCT FROM :val
""")

	# •	same value → skip
	# •	different or NULL → update

with engine.begin() as conn:  # auto-commit / rollback
    for r in dup_rows:  # rows = list[Row]
        result = conn.execute(
            update_sql,
            {
                "val": float(r.val),
                "station_id": r.station_id,
                "variable_name": r.variable_name,
                "obs_time": r.time,
            }
        )

        if result.rowcount == 0:
            print(
                f"⚠️ No row updated: "
                f"{r.station_id}, {r.variable_name}, {r.time}"
            )
        else:
            print(
                f"✅ Updated: "
                f"{r.station_id}, {r.variable_name}, {r.time}"
            )

✅ Updated: AtlSch, TempC, 2010-08-20 19:00:00
✅ Updated: AtlSch, TempC, 2010-08-20 20:00:00
✅ Updated: AtlSch, TempC, 2010-08-20 21:00:00
✅ Updated: AtlSch, TempC, 2010-08-20 22:00:00
✅ Updated: AtlSch, TempC, 2010-08-20 23:00:00
✅ Updated: AtlSch, TempC, 2010-08-21 00:00:00
✅ Updated: AtlSch, TempC, 2010-08-21 01:00:00
✅ Updated: AtlSch, TempC, 2010-08-21 02:00:00
✅ Updated: AtlSch, TempC, 2010-08-21 03:00:00
✅ Updated: AtlSch, TempC, 2010-08-21 04:00:00
✅ Updated: AtlSch, TempC, 2010-08-21 05:00:00
✅ Updated: AtlSch, TempC, 2010-08-21 06:00:00
✅ Updated: AtlSch, TempC, 2010-08-21 07:00:00
✅ Updated: AtlSch, TempC, 2010-08-21 08:00:00
✅ Updated: AtlSch, TempC, 2010-08-21 09:00:00
✅ Updated: AtlSch, TempC, 2010-08-21 10:00:00
✅ Updated: AtlSch, TempC, 2010-08-21 11:00:00
✅ Updated: AtlSch, TempC, 2010-08-21 12:00:00
✅ Updated: AtlSch, TempC, 2010-08-21 13:00:00
✅ Updated: AtlSch, TempC, 2010-08-21 14:00:00
✅ Updated: AtlSch, TempC, 2010-08-21 15:00:00
✅ Updated: AtlSch, TempC, 2010-08-

KeyboardInterrupt: 

#### batch update

In [109]:
def chunked(lst, size):
    for i in range(0, len(lst), size):
        yield i, lst[i:i + size]

BATCH_SIZE = 2000
total = len(dup_rows)

for start, chunk in chunked(dup_rows, BATCH_SIZE):
    params = [
        {
            "val": float(r.val),
            "station_id": r.station_id,
            "variable_name": r.variable_name,
            "obs_time": r.time,
        }
        for r in chunk
    ]

    with engine.begin() as conn:
        result = conn.execute(update_sql, params)

    end = min(start + BATCH_SIZE, total)
    print(
        f"✅ Updated batch {start}-{end - 1} "
        f"({result.rowcount} rows updated)"
    )

✅ Updated batch 0-1999 (1998 rows updated)
✅ Updated batch 2000-3999 (1979 rows updated)
✅ Updated batch 4000-5999 (1982 rows updated)
✅ Updated batch 6000-7999 (1989 rows updated)
✅ Updated batch 8000-9999 (1999 rows updated)
✅ Updated batch 10000-11999 (2000 rows updated)
✅ Updated batch 12000-13999 (2000 rows updated)
✅ Updated batch 14000-15999 (2000 rows updated)
✅ Updated batch 16000-17999 (2000 rows updated)
✅ Updated batch 18000-19999 (2000 rows updated)
✅ Updated batch 20000-21999 (1999 rows updated)
✅ Updated batch 22000-23999 (2000 rows updated)
✅ Updated batch 24000-25999 (2000 rows updated)
✅ Updated batch 26000-27999 (1999 rows updated)
✅ Updated batch 28000-29999 (2000 rows updated)
✅ Updated batch 30000-31999 (2000 rows updated)
✅ Updated batch 32000-33999 (1999 rows updated)
✅ Updated batch 34000-35999 (2000 rows updated)
✅ Updated batch 36000-37999 (2000 rows updated)
✅ Updated batch 38000-39999 (2000 rows updated)
✅ Updated batch 40000-41999 (2000 rows updated)
✅ Upd

### Validation

In [110]:
check_sql = sa.text("""
SELECT o.datum
FROM obs_raw AS o
JOIN meta_history AS m ON o.history_id = m.history_id
JOIN meta_station AS s ON m.station_id = s.station_id
JOIN meta_vars AS v ON o.vars_id = v.vars_id
WHERE s.native_id = :station_id
  AND v.net_var_name = :variable_name
  AND o.obs_time = :obs_time
""")

with engine.begin() as conn:
    for r in dup_rows:
        # --- UPDATE ---
        result = conn.execute(
            update_sql,
            {
                "val": float(r.val),
                "station_id": r.station_id,
                "variable_name": r.variable_name,
                "obs_time": r.time,
            }
        )

        # --- CHECK ---
        row = conn.execute(
            check_sql,
            {
                "station_id": r.station_id,
                "variable_name": r.variable_name,
                "obs_time": r.time,
            }
        ).fetchone()

        db_val = row[0] if row else None

        # --- COMPARE ---
        if row is None:
            print(f"❌ Row not found in DB: {r.station_id}, {r.variable_name}, {r.time}")
        elif db_val == float(r.val):
            print(
                f"✅ Verified: "
                f"{r.station_id}, {r.variable_name}, {r.time} "
                f"DB={db_val}"
            )
        else:
            print(
                f"⚠️ Mismatch: "
                f"{r.station_id}, {r.variable_name}, {r.time} "
                f"DB={db_val}, Row={r.val}"
            )

✅ Verified: AtlSch, TempC, 2010-08-20 19:00:00 DB=10.271
✅ Verified: AtlSch, TempC, 2010-08-20 20:00:00 DB=9.262
✅ Verified: AtlSch, TempC, 2010-08-20 21:00:00 DB=7.293
✅ Verified: AtlSch, TempC, 2010-08-20 22:00:00 DB=6.179
✅ Verified: AtlSch, TempC, 2010-08-20 23:00:00 DB=5.847
✅ Verified: AtlSch, TempC, 2010-08-21 00:00:00 DB=5.102
✅ Verified: AtlSch, TempC, 2010-08-21 01:00:00 DB=4.402
✅ Verified: AtlSch, TempC, 2010-08-21 02:00:00 DB=4.402
✅ Verified: AtlSch, TempC, 2010-08-21 03:00:00 DB=4.454
✅ Verified: AtlSch, TempC, 2010-08-21 04:00:00 DB=4.818
✅ Verified: AtlSch, TempC, 2010-08-21 05:00:00 DB=5.076
✅ Verified: AtlSch, TempC, 2010-08-21 06:00:00 DB=5.36
✅ Verified: AtlSch, TempC, 2010-08-21 07:00:00 DB=5.796
✅ Verified: AtlSch, TempC, 2010-08-21 08:00:00 DB=7.242
✅ Verified: AtlSch, TempC, 2010-08-21 09:00:00 DB=8.22
✅ Verified: AtlSch, TempC, 2010-08-21 10:00:00 DB=9.312
✅ Verified: AtlSch, TempC, 2010-08-21 11:00:00 DB=9.854
✅ Verified: AtlSch, TempC, 2010-08-21 12:00:00 DB

KeyboardInterrupt: 

In [108]:
params = [
    {
        "val": float(r.val),
        "station_id": r.station_id,
        "variable_name": r.variable_name,
        "obs_time": r.time,
    }
    for r in dup_rows
]

with engine.begin() as conn:
    result = conn.execute(update_sql, params)
    print(f"✅ Batch update executed, rowcount={result.rowcount}")


KeyboardInterrupt: 